# 📊 Dashboard Interactivo — Capacitaciones DGA (1er Semestre 2026)

Este notebook está pensado para **Google Colab**. Lee el archivo Excel, limpia los datos (hay algunas
inconsistencias típicas de formularios: mayúsculas/minúsculas mezcladas, valores fuera de rango, etc.)
y genera un dashboard interactivo con filtros (Trimestre, Modalidad, Departamento, Sector) usando
`plotly` + `ipywidgets`.

**Cómo usarlo:**
1. Ejecuta las celdas en orden (Entorno de ejecución → Ejecutar todas).
2. En la celda de carga de datos, sube el archivo `Capacitaciones_DGA_1er_Semestre_2026.xlsx` cuando se te solicite.
3. Al final, usa los menús desplegables para filtrar el dashboard.

> ℹ️ **Nota sobre Google AI Studio:** AI Studio está pensado para experimentar con modelos Gemini
> (prompts, generación de texto/imágenes), no para ejecutar notebooks de análisis de datos como este.
> Si quieres algo visualizable ahí (o en cualquier navegador sin Colab), al final del notebook te dejo
> cómo exportar este mismo dashboard como un archivo **HTML autocontenido** que se abre en cualquier navegador.


## 1. Instalar / importar librerías

In [ ]:
# En Colab, plotly e ipywidgets ya vienen preinstalados normalmente.
# Esta línea asegura tener las versiones correctas.
!pip install -q plotly ipywidgets pandas openpyxl

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

pd.set_option('display.max_columns', None)


## 2. Cargar el archivo Excel

In [ ]:
# Opción A: subir el archivo manualmente desde tu computadora (recomendado en Colab)
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

FILE_NAME = "Capacitaciones_DGA_1er_Semestre_2026.xlsx"

import os
if IN_COLAB and not os.path.exists(FILE_NAME):
    print("Sube el archivo Excel cuando se abra el diálogo...")
    uploaded = files.upload()
    FILE_NAME = list(uploaded.keys())[0]

# Opción B: si prefieres usar Google Drive, descomenta estas líneas:
# from google.colab import drive
# drive.mount('/content/drive')
# FILE_NAME = "/content/drive/MyDrive/ruta/Capacitaciones_DGA_1er_Semestre_2026.xlsx"

df_raw = pd.read_excel(FILE_NAME, sheet_name="Capacitaciones")
print(f"Filas: {df_raw.shape[0]} | Columnas: {df_raw.shape[1]}")
df_raw.head()


## 3. Limpieza de datos

El archivo tiene inconsistencias típicas de un formulario con muchos registros (2,195 filas):
- `Género`: mezcla mayúsculas/minúsculas y algunos números de celular mal ingresados en esa columna.
- `Nivel de Gobierno` y `Departamento`: mayúsculas/minúsculas inconsistentes (ej. "JUNIN" vs "Junín").
- `Edad`: algunos registros tienen "Masculino"/"Femenino"/"-" en vez de un rango de edad (error de digitación).

La limpieza de abajo **no inventa datos**: normaliza texto (may/min, espacios) y convierte valores
claramente inválidos a `"No especificado"` para que no distorsionen los gráficos.


In [ ]:
df = df_raw.copy()

def clean_text(s):
    if pd.isna(s):
        return "No especificado"
    s = str(s).strip()
    if s in ("-", "", "nan"):
        return "No especificado"
    return s

# --- Género: normalizar y descartar valores inválidos (ej. números de celular) ---
def clean_genero(g):
    g = clean_text(g).upper()
    if g.startswith("MASC"):
        return "Masculino"
    if g.startswith("FEM"):
        return "Femenino"
    return "No especificado"

df["Género"] = df["Género"].apply(clean_genero)

# --- Nivel de Gobierno: normalizar capitalización y agrupar variantes ---
def clean_nivel_gob(v):
    v = clean_text(v)
    mapping = {
        "LOCAL": "Local", "REGIONAL": "Regional", "NACIONAL": "Nacional",
    }
    return mapping.get(v.upper(), v.title() if v not in ("No especificado",) else v)

df["Nivel de Gobierno (de la entidad)"] = df["Nivel de Gobierno (de la entidad)"].apply(clean_nivel_gob)

# --- Departamento: normalizar may/min (mantiene tildes tal como vienen) ---
df["Departamento (donde se ubica la entidad - UE)"] = (
    df["Departamento (donde se ubica la entidad - UE)"].apply(clean_text).str.title()
)
# Unificar duplicados por falta de tilde (ajusta si detectas más casos)
dep_fix = {"Junin": "Junín", "Ica ": "Ica"}
df["Departamento (donde se ubica la entidad - UE)"] = df["Departamento (donde se ubica la entidad - UE)"].replace(dep_fix)

# --- Edad: descartar valores que no son un rango de edad válido ---
rangos_validos = [
    "Entre 20 y 30 años", "Entre 31 y 40 años", "Entre 41 y 50 años",
    "Entre 51 y 60 años", "Mayor a 61 años", "No deseo proporcionar mi rango de edad",
]
df["Edad"] = df["Edad"].apply(lambda x: x if x in rangos_validos else "No especificado")

# --- Modalidad de contratación: agrupar variantes de escritura ---
def clean_contrato(v):
    v = clean_text(v)
    mapping = {
        "D.L. 276": "DL 276", "D.L. 728": "DL 728", "D.L. 1057": "DL 1057",
        "Ley 30057 (Ley Servir)": "SERVIR",
    }
    return mapping.get(v, v)

df["¿CUÁL ES SU MODALIDAD DE CONTRATACIÓN EN LA ENTIDAD?"] = (
    df["¿CUÁL ES SU MODALIDAD DE CONTRATACIÓN EN LA ENTIDAD?"].apply(clean_contrato)
)

# --- Fecha: asegurar tipo datetime y crear columna de mes ---
df["FECHA DE REALIZACIÓN"] = pd.to_datetime(df["FECHA DE REALIZACIÓN"], errors="coerce")
df["Mes"] = df["FECHA DE REALIZACIÓN"].dt.to_period("M").astype(str)

print("Limpieza completa. Ejemplo de valores únicos tras limpiar 'Género':", df["Género"].unique())
print("Valores únicos 'Nivel de Gobierno':", df["Nivel de Gobierno (de la entidad)"].unique())


## 4. KPIs generales

In [ ]:
def mostrar_kpis(data):
    total_registros = len(data)
    personas_unicas = data["DNI"].nunique()
    entidades_unicas = data["Nombre de la entidad (Unidad Ejecutora)"].nunique()
    pct_virtual = (data["MODALIDAD (PRESENCIAL, VIRTUAL)"] == "VIRTUAL").mean() * 100

    kpi_html = f"""
    <div style="display:flex; gap:16px; font-family:Arial;">
      <div style="flex:1; background:#1f4e79; color:white; padding:16px; border-radius:10px; text-align:center;">
        <div style="font-size:28px; font-weight:bold;">{total_registros:,}</div>
        <div style="font-size:13px;">Participaciones registradas</div>
      </div>
      <div style="flex:1; background:#2e7d32; color:white; padding:16px; border-radius:10px; text-align:center;">
        <div style="font-size:28px; font-weight:bold;">{personas_unicas:,}</div>
        <div style="font-size:13px;">Personas únicas (DNI)</div>
      </div>
      <div style="flex:1; background:#b8860b; color:white; padding:16px; border-radius:10px; text-align:center;">
        <div style="font-size:28px; font-weight:bold;">{entidades_unicas:,}</div>
        <div style="font-size:13px;">Entidades (UE) alcanzadas</div>
      </div>
      <div style="flex:1; background:#6a1b9a; color:white; padding:16px; border-radius:10px; text-align:center;">
        <div style="font-size:28px; font-weight:bold;">{pct_virtual:.0f}%</div>
        <div style="font-size:13px;">Modalidad virtual</div>
      </div>
    </div>
    """
    display(HTML(kpi_html))

mostrar_kpis(df)


## 5. Dashboard interactivo con filtros

In [ ]:
# --- Widgets de filtro ---
trimestre_w = widgets.Dropdown(
    options=["Todos"] + sorted(df["Trimestre"].dropna().unique().tolist()),
    value="Todos", description="Trimestre:"
)
modalidad_w = widgets.Dropdown(
    options=["Todas"] + sorted(df["MODALIDAD (PRESENCIAL, VIRTUAL)"].dropna().unique().tolist()),
    value="Todas", description="Modalidad:"
)
depto_w = widgets.Dropdown(
    options=["Todos"] + sorted(df["Departamento (donde se ubica la entidad - UE)"].dropna().unique().tolist()),
    value="Todos", description="Departamento:"
)
sector_w = widgets.Dropdown(
    options=["Todos"] + sorted(df["¿A qué sector pertenece su entidad?"].dropna().unique().tolist()),
    value="Todos", description="Sector:"
)

out = widgets.Output()

def filtrar(data):
    d = data.copy()
    if trimestre_w.value != "Todos":
        d = d[d["Trimestre"] == trimestre_w.value]
    if modalidad_w.value != "Todas":
        d = d[d["MODALIDAD (PRESENCIAL, VIRTUAL)"] == modalidad_w.value]
    if depto_w.value != "Todos":
        d = d[d["Departamento (donde se ubica la entidad - UE)"] == depto_w.value]
    if sector_w.value != "Todos":
        d = d[d["¿A qué sector pertenece su entidad?"] == sector_w.value]
    return d

def actualizar(*args):
    with out:
        clear_output(wait=True)
        d = filtrar(df)

        if len(d) == 0:
            print("No hay registros para esta combinación de filtros.")
            return

        mostrar_kpis(d)

        # Participaciones por mes
        serie_mes = d.groupby("Mes").size().reset_index(name="Participaciones").sort_values("Mes")
        fig1 = px.line(serie_mes, x="Mes", y="Participaciones", markers=True,
                        title="Participaciones por mes")
        fig1.update_layout(height=350)
        fig1.show()

        col1, col2 = d, d  # reutilizamos d

        # Top 10 departamentos
        top_dep = (d["Departamento (donde se ubica la entidad - UE)"]
                   .value_counts().head(10).sort_values())
        fig2 = px.bar(top_dep, x=top_dep.values, y=top_dep.index, orientation="h",
                      title="Top 10 departamentos por participaciones",
                      labels={"x": "Participaciones", "y": ""})
        fig2.update_layout(height=400)
        fig2.show()

        # Modalidad (pie)
        fig3 = px.pie(d, names="MODALIDAD (PRESENCIAL, VIRTUAL)",
                      title="Distribución por modalidad", hole=0.4)
        fig3.update_layout(height=350)
        fig3.show()

        # Sector
        top_sector = d["¿A qué sector pertenece su entidad?"].value_counts().head(10).sort_values()
        fig4 = px.bar(top_sector, x=top_sector.values, y=top_sector.index, orientation="h",
                      title="Participaciones por sector (top 10)",
                      labels={"x": "Participaciones", "y": ""})
        fig4.update_layout(height=400)
        fig4.show()

        # Nivel de gobierno
        fig5 = px.pie(d, names="Nivel de Gobierno (de la entidad)",
                      title="Distribución por nivel de gobierno", hole=0.4)
        fig5.update_layout(height=350)
        fig5.show()

        # Género
        fig6 = px.pie(d, names="Género", title="Distribución por género", hole=0.4)
        fig6.update_layout(height=350)
        fig6.show()

        # Top 10 temáticas
        top_tema = d["TEMÁTICA DEL CURSO O TALLER"].value_counts().head(10).sort_values()
        fig7 = px.bar(top_tema, x=top_tema.values, y=[t[:60] + "..." if len(t) > 60 else t for t in top_tema.index],
                      orientation="h", title="Top 10 temáticas más dictadas",
                      labels={"x": "Participaciones", "y": ""})
        fig7.update_layout(height=450)
        fig7.show()

for w in (trimestre_w, modalidad_w, depto_w, sector_w):
    w.observe(actualizar, names="value")

filtros_box = widgets.HBox([trimestre_w, modalidad_w, depto_w, sector_w])
display(filtros_box, out)
actualizar()


## 6. (Opcional) Exportar como HTML autocontenido

Si quieres compartir el dashboard sin depender de Colab (por ejemplo, abrirlo directamente en un
navegador, adjuntarlo a un correo, o subirlo a Google AI Studio como referencia visual), puedes
generar una versión estática en HTML con los gráficos principales.


In [ ]:
from plotly.subplots import make_subplots
import plotly.io as pio

def exportar_html(data, filename="dashboard_capacitaciones.html"):
    figs = []

    serie_mes = data.groupby("Mes").size().reset_index(name="Participaciones").sort_values("Mes")
    figs.append(px.line(serie_mes, x="Mes", y="Participaciones", markers=True, title="Participaciones por mes"))

    top_dep = data["Departamento (donde se ubica la entidad - UE)"].value_counts().head(10).sort_values()
    figs.append(px.bar(top_dep, x=top_dep.values, y=top_dep.index, orientation="h",
                        title="Top 10 departamentos", labels={"x": "Participaciones", "y": ""}))

    figs.append(px.pie(data, names="MODALIDAD (PRESENCIAL, VIRTUAL)", title="Modalidad", hole=0.4))
    figs.append(px.pie(data, names="Nivel de Gobierno (de la entidad)", title="Nivel de gobierno", hole=0.4))

    with open(filename, "w", encoding="utf-8") as f:
        f.write("<html><head><meta charset='utf-8'><title>Dashboard Capacitaciones DGA</title></head><body>")
        f.write("<h1 style='font-family:Arial;'>Dashboard Capacitaciones DGA - 1er Semestre 2026</h1>")
        for fig in figs:
            f.write(pio.to_html(fig, full_html=False, include_plotlyjs="cdn"))
        f.write("</body></html>")
    print(f"Archivo generado: {filename}")
    if IN_COLAB:
        files.download(filename)

exportar_html(df)
